In [12]:
import os
import requests
import numpy as np
import cv2
from PIL import Image

In [13]:
if not os.path.exists("../data/eskimo_dog.jpg"):
    url = "https://raw.githubusercontent.com/jigsawpieces/dog-api-images/main/eskimo/n02109961_10021.jpg"
    response = requests.get(url, stream=True)

    with open("../data/eskimo_dog.jpg", "wb") as f:
        f.write(response.content)

img_pil = Image.open("../data/eskimo_dog.jpg")
img_cv2 = cv2.imread("../data/eskimo_dog.jpg")

np.array(img_pil).shape,  img_cv2.shape          # (H, W, C) for both images

((398, 500, 3), (398, 500, 3))

In [14]:
try:
    np.testing.assert_allclose(img_cv2, img_pil, atol=01e-08)
except AssertionError as e:
    print("Images are not equal:", e)


Images are not equal: 
Not equal to tolerance rtol=1e-07, atol=1e-08

Mismatched elements: 396408 / 597000 (66.4%)
Max absolute difference: 67
Max relative difference: 67.
 x: array([[[ 59, 116,  91],
        [ 58, 115,  90],
        [ 57, 114,  89],...
 y: array([[[ 91, 116,  59],
        [ 90, 115,  58],
        [ 89, 114,  57],...


**NB!:** заметим, что 0-й и 2-й каналы поменяны местами: RGB versus GBR

In [15]:
img_cv2pil = cv2.cvtColor(img_cv2, cv2.COLOR_BGR2RGB)                # BGR -> RGB 

In [16]:
try:
    np.testing.assert_allclose(img_cv2pil, img_pil, atol=01e-08)
    print("Images are equal")
except AssertionError as e:
    print("Images are not equal:", e)


Images are equal


**NB!:** из-за разных алгоритмов масштабирования и их реализации мы не всегда можем получить одинаковые результаты, вызывая в общем схожие методы. Расхождения обычно небольшие, так что явных артефактов на изображении не будет. Но лучше всё-таки помнить про это, фиксировать один алгоритм ремасштабирования и в дальнейшем придерживаться его.

In [17]:
shape = (224, 224)
img_pil = img_pil.resize(shape)
img_cv2 = cv2.resize(img_cv2, shape)

In [18]:
img_cv2pil = cv2.cvtColor(img_cv2, cv2.COLOR_BGR2RGB)     

In [19]:
try:
    np.testing.assert_allclose(img_cv2pil, img_pil, atol=01e-08)
    print("Images are equal")
except AssertionError as e:
    print("Images are not equal:", e)

Images are not equal: 
Not equal to tolerance rtol=1e-07, atol=1e-08

Mismatched elements: 94421 / 150528 (62.7%)
Max absolute difference: 56
Max relative difference: 3.
 x: array([[[ 91, 116,  58],
        [ 88, 113,  56],
        [ 89, 114,  57],...
 y: array([[[ 90, 115,  58],
        [ 88, 113,  56],
        [ 89, 114,  57],...


**NB!:** также могут не совпасть и значения пикселей

In [20]:
img_pil = np.array(img_pil) / 255
img_cv2 = img_cv2 / 255

print(np.min(img_cv2), np.min(img_pil), np.max(img_cv2), np.max(img_pil)) 

0.0 0.0 0.9490196078431372 0.9529411764705882


# Задание 1
Используя cv2, выполните:
* Чтение картинки, загруженной в примере выше.
* Преобразование в размерность 356 на 356 пикселей.
* Приведение к диапазону от 0 до 1.
* Поканальную нормализацию.
* При выполнении поканальной нормализации каждый канал (RGB), должен быть нормализован по своим значениям из массивов mean и std.  
* Выведите первый элемент второго цветового канала из преобразованного изображения, округлите до 3-го знака.
* Для поканальной нормализации используйте формулу (img - mean) / std.
* Для вывода элемента на экран предварительно посмотрите на атрибут shape полученного изображения.

In [21]:
mean = [0.4, 0.42, 0.44]
std = [0.2, 0.21, 0.22]

img = cv2.imread("../data/eskimo_dog.jpg")
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

shape = (356, 356)
img = cv2.resize(img, shape)

img = img / 255
img = (img - np.array(mean)) / np.array(std)

print(img[0, 0, 1].round(3)) 

0.166


**NB!:** При загрузке порядок размерностей H × W × C различается от того, какой требует PyTorch при использовании моделей — C × H × W.

Чтобы преобразовать изображения к формату Channels × Heigth × Width, можно воспользоваться стандартной перестановкой каналов из NumPy (так как cv2 использует numpy формат массивов для изображений - в отличие от pillow) - методом transpose: 

In [22]:
print("Размеры до: ", img.shape)

# перемещаем каналы на первую позицию
img = img.transpose(2, 0, 1)
print("Размеры после: ",img.shape) 

Размеры до:  (356, 356, 3)
Размеры после:  (3, 356, 356)
